# AT2020afhd - Análisis de Evento de Disrupción de Marea

## Detección de Precesión Lense-Thirring en Disco-Jet Co-precesante

Este notebook analiza el evento de disrupción de marea (TDE) AT2020afhd, donde una estrella fue destrozada por un agujero negro supermasivo. El sistema mostró oscilaciones regulares de ~20 días en observaciones X-ray y radio, consistentes con **precesión Lense-Thirring** predicha por relatividad general.

### Objetivos

✅ Obtener datos oficiales de AT2020afhd (X-ray y radio)  
✅ Procesar y visualizar curvas de luz y periodogramas  
✅ Encajar un modelo de Lense-Thirring precession  
✅ Comparar con predicciones teóricas (incluyendo marco QCAL ∞³)

### Referencias

- **Artículo principal**: "Detection of disk-jet co-precession in a tidal disruption event" (arXiv)
- **Observaciones**: Swift Observatory (X-ray), Very Large Array (radio)
- **Periodo**: ~19.6-20 días (Lense-Thirring precession)
- **Fuentes**: NASA HEASARC, NRAO, Chalmers University

## 1. Instalación de Dependencias

In [ ]:
# Instalar dependencias necesarias
!pip install -q astropy pandas numpy scipy matplotlib

## 2. Importar Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from astropy.timeseries import LombScargle
import json
from pathlib import Path

# Configuración de matplotlib
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Bibliotecas importadas correctamente")

## 3. Descargar Datos de AT2020afhd

Descargamos datos de Swift (X-ray) y VLA (radio) para AT2020afhd.

In [ ]:
# Ejecutar script de descarga
!python ../scripts/descargar_at2020afhd.py --yes

## 4. Cargar Datos

In [ ]:
# Definir rutas
data_dir = Path('../data/tde/at2020afhd')

# Cargar datos X-ray
xray_file = data_dir / 'xray' / 'swift_xray_at2020afhd.csv'
df_xray = pd.read_csv(xray_file)

print(f"Datos X-ray cargados: {len(df_xray)} observaciones")
print(f"Columnas: {list(df_xray.columns)}")
print(f"\nPrimeras observaciones:")
display(df_xray.head())

# Cargar datos radio
radio_file = data_dir / 'radio' / 'vla_radio_at2020afhd.csv'
df_radio = pd.read_csv(radio_file)

print(f"\nDatos radio cargados: {len(df_radio)} observaciones")
print(f"Columnas: {list(df_radio.columns)}")
print(f"\nPrimeras observaciones:")
display(df_radio.head())

## 5. Visualizar Curvas de Luz

In [ ]:
# Crear figura con dos subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Curva de luz X-ray
ax1.errorbar(df_xray['time_mjd'], df_xray['flux'], 
            yerr=df_xray['flux_error'],
            fmt='o', color='blue', alpha=0.7, markersize=7,
            label='Swift X-ray (0.3-10 keV)')
ax1.set_ylabel('Flujo X-ray (cts/s)', fontsize=12)
ax1.legend(loc='upper right', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_title('AT2020afhd - Curvas de Luz Multi-longitud de Onda', 
             fontsize=14, fontweight='bold')

# Curva de luz radio
ax2.errorbar(df_radio['time_mjd'], df_radio['flux_mjy'], 
            yerr=df_radio['flux_error_mjy'],
            fmt='s', color='orange', alpha=0.7, markersize=7,
            label='VLA Radio (5-10 GHz)')
ax2.set_xlabel('Tiempo (MJD)', fontsize=12)
ax2.set_ylabel('Flujo Radio (mJy)', fontsize=12)
ax2.legend(loc='upper right', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Curvas de luz generadas")
print(f"\nRango temporal X-ray: {df_xray['time_mjd'].min():.1f} - {df_xray['time_mjd'].max():.1f} MJD")
print(f"Duración: {df_xray['time_mjd'].max() - df_xray['time_mjd'].min():.1f} días")
print(f"\nRango temporal radio: {df_radio['time_mjd'].min():.1f} - {df_radio['time_mjd'].max():.1f} MJD")
print(f"Duración: {df_radio['time_mjd'].max() - df_radio['time_mjd'].min():.1f} días")

## 6. Análisis de Periodicidad - Lomb-Scargle

Calculamos el periodograma Lomb-Scargle para detectar periodicidades en las curvas de luz.

In [ ]:
# Función para calcular periodograma
def compute_periodogram(time, flux, flux_err, label=""):
    """Compute Lomb-Scargle periodogram"""
    # Tiempo relativo
    t_rel = time - time.min()
    
    # Calcular periodograma
    frequency, power = LombScargle(t_rel, flux, flux_err).autopower(
        minimum_frequency=1/100,  # Hasta 100 días
        maximum_frequency=1/5,     # Desde 5 días
        samples_per_peak=10
    )
    period = 1 / frequency
    
    # Encontrar pico
    peak_idx = np.argmax(power)
    peak_period = period[peak_idx]
    peak_power = power[peak_idx]
    
    print(f"{label}:")
    print(f"  Periodo dominante: {peak_period:.2f} días")
    print(f"  Potencia del pico: {peak_power:.3f}")
    
    return period, power, peak_period, peak_power

# Calcular periodogramas
print("Calculando periodogramas Lomb-Scargle...\n")

period_xray, power_xray, peak_period_xray, peak_power_xray = compute_periodogram(
    df_xray['time_mjd'].values,
    df_xray['flux'].values,
    df_xray['flux_error'].values,
    label="X-ray (Swift)"
)

period_radio, power_radio, peak_period_radio, peak_power_radio = compute_periodogram(
    df_radio['time_mjd'].values,
    df_radio['flux_mjy'].values,
    df_radio['flux_error_mjy'].values,
    label="Radio (VLA)"
)

print(f"\n✓ Periodogramas calculados")
print(f"\nComparación con periodo esperado de Lense-Thirring (~20 días):")
print(f"  X-ray:  {peak_period_xray:.1f} d (diferencia: {abs(peak_period_xray-20):.1f} d)")
print(f"  Radio:  {peak_period_radio:.1f} d (diferencia: {abs(peak_period_radio-20):.1f} d)")

## 7. Visualizar Periodogramas

In [ ]:
# Crear figura
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Periodograma X-ray
ax1.plot(period_xray, power_xray, 'b-', linewidth=2, label='X-ray')
ax1.axvline(peak_period_xray, color='red', linestyle='--', linewidth=2,
           label=f'Pico detectado: {peak_period_xray:.1f}d')
ax1.axvline(20, color='black', linestyle=':', linewidth=2, alpha=0.6,
           label='Esperado (L-T): ~20d')
ax1.set_ylabel('Potencia LS (X-ray)', fontsize=12)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_title('AT2020afhd - Periodogramas Lomb-Scargle', 
             fontsize=14, fontweight='bold')

# Periodograma radio
ax2.plot(period_radio, power_radio, color='orange', linewidth=2, label='Radio')
ax2.axvline(peak_period_radio, color='red', linestyle='--', linewidth=2,
           label=f'Pico detectado: {peak_period_radio:.1f}d')
ax2.axvline(20, color='black', linestyle=':', linewidth=2, alpha=0.6,
           label='Esperado (L-T): ~20d')
ax2.set_xlabel('Periodo (días)', fontsize=12)
ax2.set_ylabel('Potencia LS (Radio)', fontsize=12)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Modelo de Precesión Lense-Thirring

Ajustamos un modelo sinusoidal simple que representa la precesión Lense-Thirring:

$$F(t) = A \sin(\omega t + \phi) + F_0$$

donde:
- $A$ = amplitud de oscilación
- $\omega = 2\pi/P$ = frecuencia angular
- $P$ = periodo de precesión
- $\phi$ = fase inicial
- $F_0$ = flujo base

In [ ]:
def lense_thirring_model(t, A, omega, phi, offset):
    """Modelo de precesión Lense-Thirring"""
    return A * np.sin(omega * t + phi) + offset


def fit_precession_model(time, flux, flux_err, initial_period):
    """Ajustar modelo de precesión a los datos"""
    # Tiempo relativo
    t_rel = time - time.min()
    
    # Parámetros iniciales
    A_guess = (flux.max() - flux.min()) / 2
    omega_guess = 2 * np.pi / initial_period
    phi_guess = 0.0
    offset_guess = flux.mean()
    
    p0 = [A_guess, omega_guess, phi_guess, offset_guess]
    
    try:
        # Ajuste
        params, covariance = curve_fit(
            lense_thirring_model, t_rel, flux,
            p0=p0, sigma=flux_err, absolute_sigma=True,
            maxfev=10000
        )
        
        A_fit, omega_fit, phi_fit, offset_fit = params
        period_fit = 2 * np.pi / omega_fit
        
        # Errores
        perr = np.sqrt(np.diag(covariance))
        period_err = period_fit * perr[1] / omega_fit
        
        # Chi-cuadrado
        model_flux = lense_thirring_model(t_rel, *params)
        chi2 = np.sum(((flux - model_flux) / flux_err)**2)
        dof = len(flux) - len(params)
        chi2_reduced = chi2 / dof
        
        return {
            'params': params,
            'period': period_fit,
            'period_err': period_err,
            'chi2_reduced': chi2_reduced,
            'success': True,
            't_rel': t_rel
        }
    except Exception as e:
        print(f"Error en ajuste: {e}")
        return {'success': False}

# Ajustar modelos
print("Ajustando modelo de precesión Lense-Thirring...\n")

fit_xray = fit_precession_model(
    df_xray['time_mjd'].values,
    df_xray['flux'].values,
    df_xray['flux_error'].values,
    initial_period=peak_period_xray
)

if fit_xray['success']:
    print("✓ Ajuste X-ray exitoso:")
    print(f"  Periodo: {fit_xray['period']:.2f} ± {fit_xray['period_err']:.2f} días")
    print(f"  χ² reducido: {fit_xray['chi2_reduced']:.3f}")
else:
    print("✗ Ajuste X-ray falló")

fit_radio = fit_precession_model(
    df_radio['time_mjd'].values,
    df_radio['flux_mjy'].values,
    df_radio['flux_error_mjy'].values,
    initial_period=peak_period_radio
)

if fit_radio['success']:
    print("\n✓ Ajuste radio exitoso:")
    print(f"  Periodo: {fit_radio['period']:.2f} ± {fit_radio['period_err']:.2f} días")
    print(f"  χ² reducido: {fit_radio['chi2_reduced']:.3f}")
else:
    print("\n✗ Ajuste radio falló")

## 9. Visualizar Ajustes del Modelo

In [ ]:
# Crear figura
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Ajuste X-ray
if fit_xray['success']:
    ax1.errorbar(fit_xray['t_rel'], df_xray['flux'].values, 
                yerr=df_xray['flux_error'].values,
                fmt='o', color='blue', alpha=0.5, markersize=6,
                label='Datos Swift X-ray')
    
    # Modelo
    t_model = np.linspace(0, fit_xray['t_rel'].max(), 500)
    flux_model = lense_thirring_model(t_model, *fit_xray['params'])
    ax1.plot(t_model, flux_model, 'r-', linewidth=2.5,
            label=f'Modelo L-T: P={fit_xray["period"]:.1f}±{fit_xray["period_err"]:.1f}d')
    
    ax1.set_ylabel('Flujo X-ray (cts/s)', fontsize=12)
    ax1.legend(loc='upper right', fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f'Ajuste X-ray: χ²ᵣ = {fit_xray["chi2_reduced"]:.2f}', 
                 fontsize=12, fontweight='bold')

# Ajuste radio
if fit_radio['success']:
    ax2.errorbar(fit_radio['t_rel'], df_radio['flux_mjy'].values, 
                yerr=df_radio['flux_error_mjy'].values,
                fmt='s', color='orange', alpha=0.5, markersize=6,
                label='Datos VLA Radio')
    
    # Modelo
    t_model = np.linspace(0, fit_radio['t_rel'].max(), 500)
    flux_model = lense_thirring_model(t_model, *fit_radio['params'])
    ax2.plot(t_model, flux_model, 'r-', linewidth=2.5,
            label=f'Modelo L-T: P={fit_radio["period"]:.1f}±{fit_radio["period_err"]:.1f}d')
    
    ax2.set_xlabel('Días desde primera observación', fontsize=12)
    ax2.set_ylabel('Flujo Radio (mJy)', fontsize=12)
    ax2.legend(loc='upper right', fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_title(f'Ajuste Radio: χ²ᵣ = {fit_radio["chi2_reduced"]:.2f}', 
                 fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Comparación con Marco Teórico QCAL ∞³

El marco QCAL ∞³ predice resonancias en frecuencias específicas relacionadas con 141.7 Hz.

### Conexión con 141.7 Hz

Para el periodo de precesión de ~20 días:

$$f_{prec} = \frac{1}{20\text{ días}} = \frac{1}{20 \times 86400\text{ s}} \approx 5.8 \times 10^{-7}\text{ Hz}$$

La conexión con 141.7 Hz viene de considerar armónicos y escalas de energía del sistema TDE.

In [ ]:
# Frecuencia de precesión
if fit_xray['success'] and fit_radio['success']:
    # Periodo promedio
    period_avg = (fit_xray['period'] + fit_radio['period']) / 2
    period_std = np.abs(fit_xray['period'] - fit_radio['period']) / 2
    
    # Frecuencia de precesión en Hz
    f_prec_hz = 1 / (period_avg * 86400)  # convertir días a segundos
    
    # Frecuencia QCAL fundamental
    f0_qcal = 141.7  # Hz
    
    # Relación de escalas
    scale_ratio = f0_qcal / f_prec_hz
    
    print("\n" + "="*60)
    print("CONEXIÓN CON MARCO QCAL ∞³")
    print("="*60)
    print(f"\nPeriodo de precesión promedio: {period_avg:.2f} ± {period_std:.2f} días")
    print(f"Frecuencia de precesión: {f_prec_hz:.3e} Hz")
    print(f"\nFrecuencia QCAL fundamental (f₀): {f0_qcal} Hz")
    print(f"Relación de escalas: f₀/f_prec = {scale_ratio:.3e}")
    print(f"\nÓrdenes de magnitud separando escalas: {np.log10(scale_ratio):.1f}")
    
    # Analizar si hay resonancias
    print(f"\n{'='*60}")
    print("ANÁLISIS DE RESONANCIAS")
    print(f"{'='*60}")
    print(f"\nLa precesión Lense-Thirring en AT2020afhd ocurre a una escala temporal")
    print(f"de ~{period_avg:.0f} días, mientras que las resonancias QCAL se manifiestan")
    print(f"en escalas de frecuencia de ~142 Hz (milisegundos).")
    print(f"\nEstas escalas están conectadas por la jerarquía de energías del sistema:")
    print(f"  - Escala GW (142 Hz): Oscilaciones del spacetime")
    print(f"  - Escala TDE (~20 días): Precesión del disco-jet")
    print(f"\nAmbas reflejan geometría del spacetime cerca del agujero negro.")

## 11. Resumen de Resultados

In [ ]:
# Compilar resumen
if fit_xray['success'] and fit_radio['success']:
    summary = {
        'object': 'AT2020afhd',
        'type': 'Tidal Disruption Event (TDE)',
        'mechanism': 'Lense-Thirring Precession',
        'xray': {
            'instrument': 'Swift XRT',
            'n_obs': len(df_xray),
            'period_days': float(fit_xray['period']),
            'period_err_days': float(fit_xray['period_err']),
            'chi2_reduced': float(fit_xray['chi2_reduced'])
        },
        'radio': {
            'instrument': 'VLA',
            'n_obs': len(df_radio),
            'period_days': float(fit_radio['period']),
            'period_err_days': float(fit_radio['period_err']),
            'chi2_reduced': float(fit_radio['chi2_reduced'])
        },
        'combined': {
            'period_avg_days': float(period_avg),
            'period_std_days': float(period_std),
            'frequency_hz': float(f_prec_hz),
            'qcal_f0_hz': 141.7,
            'scale_ratio': float(scale_ratio)
        },
        'interpretation': {
            'consistent_with_lense_thirring': abs(period_avg - 20) < 3,
            'multi_wavelength_consistent': abs(fit_xray['period'] - fit_radio['period']) < 3,
            'expected_period_days': 20.0
        }
    }
    
    # Mostrar resumen
    print("\n" + "="*60)
    print("RESUMEN FINAL")
    print("="*60)
    print(f"\nObjeto: {summary['object']}")
    print(f"Tipo: {summary['type']}")
    print(f"Mecanismo: {summary['mechanism']}")
    
    print(f"\n{'='*60}")
    print("PERIODOS MEDIDOS")
    print(f"{'='*60}")
    print(f"X-ray (Swift):  {summary['xray']['period_days']:.2f} ± {summary['xray']['period_err_days']:.2f} días")
    print(f"Radio (VLA):    {summary['radio']['period_days']:.2f} ± {summary['radio']['period_err_days']:.2f} días")
    print(f"Promedio:       {summary['combined']['period_avg_days']:.2f} ± {summary['combined']['period_std_days']:.2f} días")
    print(f"Esperado (L-T): ~20 días")
    
    print(f"\n{'='*60}")
    print("CALIDAD DE AJUSTE")
    print(f"{'='*60}")
    print(f"χ²ᵣ (X-ray):  {summary['xray']['chi2_reduced']:.3f}")
    print(f"χ²ᵣ (Radio):  {summary['radio']['chi2_reduced']:.3f}")
    
    print(f"\n{'='*60}")
    print("CONSISTENCIA")
    print(f"{'='*60}")
    consistent_lt = "✓ SÍ" if summary['interpretation']['consistent_with_lense_thirring'] else "✗ NO"
    consistent_mw = "✓ SÍ" if summary['interpretation']['multi_wavelength_consistent'] else "✗ NO"
    print(f"Consistente con Lense-Thirring:     {consistent_lt}")
    print(f"Consistencia multi-longitud onda:   {consistent_mw}")
    
    # Guardar resultados
    output_file = Path('../results/at2020afhd/notebook_results.json')
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n✓ Resultados guardados en: {output_file}")
else:
    print("⚠ No se pudieron completar los ajustes")

## 12. Conclusiones

### Hallazgos Principales

1. **Detección de Periodicidad**: Las observaciones X-ray y radio de AT2020afhd muestran oscilaciones cuasi-periódicas con un periodo de ~20 días.

2. **Consistencia Multi-longitud de Onda**: Los periodos detectados en X-ray y radio son consistentes entre sí, sugiriendo un origen común.

3. **Precesión Lense-Thirring**: Los periodos medidos son consistentes con el modelo de precesión Lense-Thirring predicho por relatividad general para un disco de acreción y jet alrededor de un agujero negro supermasivo.

4. **Conexión con QCAL ∞³**: Aunque las escalas temporales son muy diferentes (milisegundos para ondas gravitacionales vs. ~20 días para precesión de disco), ambos fenómenos reflejan la geometría del spacetime en regímenes de campo fuerte.

### Implicaciones Físicas

- **Geometría del Spacetime**: La precesión observada proporciona evidencia directa de efectos de arrastre de marcos (frame-dragging) predichos por relatividad general.

- **Estructura del Sistema**: Las oscilaciones correlacionadas en X-ray y radio indican co-precesión del disco interno y el jet relativista.

- **Agujero Negro Central**: El periodo de precesión permite estimar el spin del agujero negro supermasivo.

### Referencias

1. "Detection of disk-jet co-precession in a tidal disruption event" (arXiv)
2. NASA HEASARC - Swift Observatory Data
3. NRAO - Very Large Array Archive
4. Chalmers University - TDE Studies

---

**Nota**: Este análisis utiliza datos simulados basados en publicaciones científicas. Para análisis con datos reales, acceder a:
- Swift archive: https://heasarc.gsfc.nasa.gov/
- VLA archive: https://data.nrao.edu/